# Qwen2.5-7B-Instruct Zero-Shot Government QA Evaluation — FIXED

This notebook evaluates **`Qwen/Qwen2.5-7B-Instruct`** in a **true zero-shot** setting on:

`/kaggle/input/datasets/akra1234/government/merged_test_data.csv`

The training file is intentionally **not used** for prompting, fine-tuning, or demonstrations.

## Fixes in this version
- Forces a **clean regeneration of every answer** by default, so an old `prediction_partial.csv` cannot keep stale/truncated answers.
- Replaces the old fixed `MAX_NEW_TOKENS = 256` generation with **adaptive deterministic generation**: 512 tokens initially, then automatic retries at 1024, 2048, and 4096 tokens only for outputs that still hit the generation limit.
- Detects truncation by checking whether generation actually reached an EOS/stop token, rather than only counting non-pad tokens.
- Saves generation audit columns (`generated_tokens`, `max_new_tokens_used`, `generation_attempts`).
- Refuses to compute final evaluation metrics if a truncated output still remains after the largest retry budget.

Outputs:
- `/kaggle/working/prediction.csv`
- `/kaggle/working/result.csv`

Metrics:
- Normalized Exact Match
- Token F1
- Fuzzy Match
- Corpus BLEU
- ROUGE-1
- ROUGE-2
- ROUGE-L
- METEOR
- BERTScore Precision
- BERTScore Recall
- BERTScore F1
- Truncated Outputs


In [ ]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# ============================================================

!pip install -q -U "transformers>=4.45" accelerate bitsandbytes \
    sacrebleu rapidfuzz nltk "bert-score==0.3.13"


In [ ]:
# ============================================================
# CELL 2 — IMPORTS + PATHS + LOAD TEST DATA
# ============================================================

import os
import re
import gc
import random
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# True zero-shot: this training file is intentionally NOT loaded or used.
TRAIN_JSONL = "/kaggle/input/datasets/akra1234/government/government_chat_train.jsonl"

TEST_CSV = "/kaggle/input/datasets/akra1234/government/merged_test_data.csv"

OUT_DIR = Path("/kaggle/working")
PRED_PATH = OUT_DIR / "prediction.csv"
PARTIAL_PATH = OUT_DIR / "prediction_partial.csv"
RESULT_PATH = OUT_DIR / "result.csv"

assert os.path.exists(TEST_CSV), f"Test file not found: {TEST_CSV}"

raw_df = pd.read_csv(TEST_CSV).fillna("")

print("Rows:", len(raw_df))
print("Columns:", raw_df.columns.tolist())


def choose_column(columns, candidates, required=True):
    lookup = {str(c).lower(): c for c in columns}
    for name in candidates:
        if name.lower() in lookup:
            return lookup[name.lower()]
    if required:
        raise ValueError(
            f"Could not find any of {candidates}. Available columns: {list(columns)}"
        )
    return None


QUESTION_COL = choose_column(
    raw_df.columns,
    ["instruction", "question", "prompt", "query"]
)

GOLD_COL = choose_column(
    raw_df.columns,
    ["output", "gold", "reference", "answer", "target"]
)

INPUT_COL = choose_column(
    raw_df.columns,
    ["input"],
    required=False
)

print("Question column:", QUESTION_COL)
print("Gold column:", GOLD_COL)
print("Optional input column:", INPUT_COL)

df = raw_df.copy()

df["question"] = df[QUESTION_COL].astype(str).str.strip()
df["gold"] = df[GOLD_COL].astype(str).str.strip()

if INPUT_COL is not None and INPUT_COL != QUESTION_COL:
    extra = df[INPUT_COL].astype(str).str.strip()
    df["model_input"] = [
        q if not x else f"{q}\n\nঅতিরিক্ত তথ্য:\n{x}"
        for q, x in zip(df["question"], extra)
    ]
else:
    df["model_input"] = df["question"]

display(df.head(3))


In [ ]:
# ============================================================
# CELL 3 — LOAD QWEN2.5-7B-INSTRUCT IN 4-BIT
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is strongly recommended. In Kaggle: Settings -> Accelerator -> GPU."
    )

compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
)

model.eval()

print("Loaded:", MODEL_NAME)
print("Device:", next(model.parameters()).device)


In [ ]:
# ============================================================
# CELL 4 — TRUE ZERO-SHOT GENERATION (FIXED)
# Clean regeneration + adaptive retry for truncated answers
# ============================================================

SYSTEM_PROMPT = (
    "আপনি বাংলাদেশের সরকারি সেবা সম্পর্কিত প্রশ্নের সহায়ক। "
    "ব্যবহারকারীর প্রশ্নের উত্তর বাংলায় দিন। "
    "উত্তরটি সংক্ষিপ্ত, সরাসরি ও তথ্যভিত্তিক রাখুন। "
    "কোনো reference answer, dataset, training example বা evaluation-এর কথা উল্লেখ করবেন না।"
)

# Initial generation is reasonably fast. Only answers that actually hit
# the token limit are regenerated with the larger budgets below.
BATCH_SIZE = 4
RETRY_BATCH_SIZE = 1
MAX_INPUT_TOKENS = 2048
GENERATION_BUDGETS = (512, 1024, 2048, 4096)

# True = ignore/delete any old Kaggle working outputs and regenerate ALL rows.
# After a clean run has started, you may set this to False only if you
# intentionally want to resume from this notebook's prediction_partial.csv.
FORCE_REGENERATE = True

# Do not silently evaluate incomplete generations.
FAIL_IF_TRUNCATED = True


def make_prompt(user_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(user_text)},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def _get_eos_token_ids():
    """Collect all EOS ids used by the model/tokenizer."""
    eos = model.generation_config.eos_token_id

    if eos is None:
        eos = tokenizer.eos_token_id

    if eos is None:
        raise ValueError("No EOS token id is configured for this model/tokenizer.")

    if isinstance(eos, int):
        eos_ids = [eos]
    else:
        eos_ids = list(eos)

    if tokenizer.eos_token_id is not None:
        eos_ids.append(int(tokenizer.eos_token_id))

    # Keep order stable while removing duplicates.
    return list(dict.fromkeys(int(x) for x in eos_ids))


EOS_TOKEN_IDS = _get_eos_token_ids()
GEN_EOS = EOS_TOKEN_IDS[0] if len(EOS_TOKEN_IDS) == 1 else EOS_TOKEN_IDS
EOS_TOKEN_ID_SET = set(EOS_TOKEN_IDS)

print("Generation EOS token ids:", EOS_TOKEN_IDS)
print("Generation budgets:", GENERATION_BUDGETS)


def generate_batch(texts, max_new_tokens):
    prompts = [make_prompt(x) for x in texts]

    batch = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    # With device_map="auto", send inputs to the model's first device.
    input_device = next(model.parameters()).device
    batch = {k: v.to(input_device) for k, v in batch.items()}

    # Causal-LM output contains the padded input followed by generated tokens.
    input_width = batch["input_ids"].shape[1]

    with torch.inference_mode():
        generated = model.generate(
            **batch,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=GEN_EOS,
        )

    new_tokens = generated[:, input_width:]
    token_rows = new_tokens.detach().cpu().tolist()

    answers = tokenizer.batch_decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    answers = [x.strip() for x in answers]

    generated_lengths = []
    truncated_flags = []

    for token_ids in token_rows:
        first_eos_position = next(
            (j for j, tok in enumerate(token_ids) if tok in EOS_TOKEN_ID_SET),
            None,
        )

        if first_eos_position is None:
            # No EOS was generated: generation stopped because max_new_tokens
            # was exhausted, so the answer is genuinely truncated.
            generated_lengths.append(len(token_ids))
            truncated_flags.append(True)
        else:
            # +1 includes the EOS token in the audit length.
            generated_lengths.append(first_eos_position + 1)
            truncated_flags.append(False)

    return answers, truncated_flags, generated_lengths


def save_partial(frame):
    frame.sort_values("row_index").to_csv(
        PARTIAL_PATH,
        index=False,
        encoding="utf-8-sig",
    )


# ------------------------------------------------------------
# Clean start / optional resume
# ------------------------------------------------------------

if FORCE_REGENERATE:
    for old_path in [PARTIAL_PATH, PRED_PATH, RESULT_PATH]:
        if old_path.exists():
            old_path.unlink()
            print("Removed old output:", old_path)

predictions = []

if (not FORCE_REGENERATE) and PARTIAL_PATH.exists():
    partial = pd.read_csv(PARTIAL_PATH).fillna("")

    required_cols = {
        "row_index", "question", "gold", "prediction", "truncated",
        "generated_tokens", "max_new_tokens_used", "generation_attempts"
    }

    if len(partial) <= len(df) and required_cols.issubset(partial.columns):
        predictions = partial.to_dict("records")
        print(f"Resuming from {len(predictions)} completed rows.")
    else:
        print("Ignoring incompatible partial file and starting fresh.")


# ------------------------------------------------------------
# Pass 1: generate EVERY answer from scratch with 512 tokens
# ------------------------------------------------------------

initial_budget = GENERATION_BUDGETS[0]
start = len(predictions)

for i in tqdm(
    range(start, len(df), BATCH_SIZE),
    desc=f"Qwen zero-shot generation ({initial_budget} tokens)"
):
    end = min(i + BATCH_SIZE, len(df))
    batch_texts = df.iloc[i:end]["model_input"].tolist()

    answers, truncated_flags, generated_lengths = generate_batch(
        batch_texts,
        max_new_tokens=initial_budget,
    )

    for local_idx, (answer, truncated, gen_len) in enumerate(
        zip(answers, truncated_flags, generated_lengths)
    ):
        row_idx = i + local_idx
        row = df.iloc[row_idx]

        record = {
            "row_index": row_idx,
            "question": row["question"],
            "gold": row["gold"],
            "prediction": answer,
            "truncated": bool(truncated),
            "generated_tokens": int(gen_len),
            "max_new_tokens_used": int(initial_budget),
            "generation_attempts": 1,
        }

        # Preserve useful metadata when available.
        for col in ["id", "domain", "topic", "question_type", "source_url", "split"]:
            if col in df.columns:
                record[col] = row[col]

        predictions.append(record)

    save_partial(pd.DataFrame(predictions))


pred_df = (
    pd.DataFrame(predictions)
    .sort_values("row_index")
    .reset_index(drop=True)
)

assert len(pred_df) == len(df), (
    f"Generated {len(pred_df)} predictions for {len(df)} test rows."
)

print(
    f"After {initial_budget}-token pass, truncated outputs:",
    int(pred_df["truncated"].astype(bool).sum())
)


# ------------------------------------------------------------
# Adaptive retries: only regenerate answers that hit the limit
# Because generation is deterministic (do_sample=False), the longer
# retry reproduces the same beginning and simply gets more room to finish.
# ------------------------------------------------------------

for retry_budget in GENERATION_BUDGETS[1:]:
    truncated_indices = pred_df.index[
        pred_df["truncated"].astype(bool)
    ].tolist()

    if not truncated_indices:
        break

    print(
        f"Retrying {len(truncated_indices)} truncated outputs "
        f"with max_new_tokens={retry_budget}..."
    )

    for pos in tqdm(
        range(0, len(truncated_indices), RETRY_BATCH_SIZE),
        desc=f"Retry at {retry_budget} tokens"
    ):
        frame_indices = truncated_indices[pos:pos + RETRY_BATCH_SIZE]
        row_indices = pred_df.loc[frame_indices, "row_index"].astype(int).tolist()
        retry_texts = df.iloc[row_indices]["model_input"].tolist()

        answers, truncated_flags, generated_lengths = generate_batch(
            retry_texts,
            max_new_tokens=retry_budget,
        )

        for frame_idx, answer, truncated, gen_len in zip(
            frame_indices, answers, truncated_flags, generated_lengths
        ):
            pred_df.at[frame_idx, "prediction"] = answer
            pred_df.at[frame_idx, "truncated"] = bool(truncated)
            pred_df.at[frame_idx, "generated_tokens"] = int(gen_len)
            pred_df.at[frame_idx, "max_new_tokens_used"] = int(retry_budget)
            pred_df.at[frame_idx, "generation_attempts"] = (
                int(pred_df.at[frame_idx, "generation_attempts"]) + 1
            )

        save_partial(pred_df)

    print(
        f"Remaining truncated after {retry_budget}-token retry:",
        int(pred_df["truncated"].astype(bool).sum())
    )


# ------------------------------------------------------------
# Final validation + save fresh predictions
# ------------------------------------------------------------

remaining_truncated = int(pred_df["truncated"].astype(bool).sum())

if remaining_truncated:
    truncated_debug = pred_df.loc[
        pred_df["truncated"].astype(bool),
        [
            "row_index", "question", "prediction",
            "generated_tokens", "max_new_tokens_used"
        ]
    ]
    debug_path = OUT_DIR / "still_truncated_after_retry.csv"
    truncated_debug.to_csv(debug_path, index=False, encoding="utf-8-sig")
    print("Saved remaining-truncation audit:", debug_path)

    if FAIL_IF_TRUNCATED:
        raise RuntimeError(
            f"{remaining_truncated} outputs are still truncated even after "
            f"max_new_tokens={GENERATION_BUDGETS[-1]}. "
            "Evaluation was stopped so incomplete answers are not scored. "
            "Inspect still_truncated_after_retry.csv or increase the final budget."
        )

pred_df.to_csv(
    PRED_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Generation complete:", len(pred_df))
print("Truncated outputs:", remaining_truncated)
print("Saved fresh predictions:", PRED_PATH)
print(
    "Maximum generation budget actually used:",
    int(pred_df["max_new_tokens_used"].max())
)

display(pred_df.head(3))


In [ ]:
# ============================================================
# CELL 5 — FREE QWEN GPU MEMORY BEFORE BERTSCORE
# ============================================================

del model

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Qwen model removed from memory.")


In [ ]:
# ============================================================
# CELL 6 — METRIC FUNCTIONS
# Keeps the same normalization / Token F1 / ROUGE definitions
# as the previous evaluation pipeline.
# ============================================================

BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)


def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(
        BN_TO_EN
    ).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokens(text):
    return normalize(text).split()


# ---------------- Normalized Exact Match ----------------

def normalized_exact_match(pred, gold):

    return float(
        normalize(pred)
        ==
        normalize(gold)
    )


# ---------------- Token F1 ----------------

def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    overlap = sum(
        (
            Counter(p)
            &
            Counter(g)
        ).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------------- ROUGE-N F1 ----------------

def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pg = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    gg = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum(
        (pg & gg).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pg.values())
    recall = overlap / sum(gg.values())

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------------- ROUGE-L F1 ----------------

def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(dp[j-1] + 1)

            else:
                new.append(
                    max(dp[j], new[-1])
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ---------------- Bengali-safe METEOR ----------------
# NLTK's default METEOR uses English Porter stemming + English WordNet.
# For Bangla evaluation, disable those English-only lexical resources while
# retaining METEOR's exact-token alignment and fragmentation penalty.

class IdentityStemmer:
    def stem(self, word):
        return word


class EmptyWordNet:
    def synsets(self, word):
        return []


IDENTITY_STEMMER = IdentityStemmer()
EMPTY_WORDNET = EmptyWordNet()


def meteor_bn(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p and not g:
        return 1.0

    if not p or not g:
        return 0.0

    return meteor_score(
        [g],
        p,
        stemmer=IDENTITY_STEMMER,
        wordnet=EMPTY_WORDNET,
    )


In [ ]:
# ============================================================
# CELL 7 — COMPUTE ROW-LEVEL METRICS
# ============================================================

eval_df = pd.read_csv(PRED_PATH).fillna("")

# Safety check: metrics must be based only on complete generations.
if "truncated" in eval_df.columns:
    n_truncated_for_eval = int(
        eval_df["truncated"].astype(str).str.lower().eq("true").sum()
    )
    if n_truncated_for_eval:
        raise RuntimeError(
            f"Refusing to evaluate {n_truncated_for_eval} truncated predictions. "
            "Re-run CELL 4 with a larger final generation budget."
        )

print(f"Evaluating {len(eval_df)} fresh, non-truncated predictions...")

eval_df["Normalized Exact Match"] = [
    normalized_exact_match(p, g)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["Token F1"] = [
    token_f1(p, g)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["ROUGE-1"] = [
    rouge_n(p, g, 1)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["ROUGE-2"] = [
    rouge_n(p, g, 2)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["ROUGE-L"] = [
    rouge_l(p, g)
    for p, g in zip(
        eval_df["prediction"],
        eval_df["gold"]
    )
]

eval_df["METEOR"] = [
    meteor_bn(p, g)
    for p, g in tqdm(
        zip(
            eval_df["prediction"],
            eval_df["gold"]
        ),
        total=len(eval_df),
        desc="METEOR"
    )
]


# ============================================================
# CORPUS BLEU
# ============================================================

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_texts = [
    " ".join(tokens(x))
    for x in eval_df["prediction"]
]

gold_texts = [
    " ".join(tokens(x))
    for x in eval_df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_texts,
        [gold_texts]
    ).score
    / 100
)

print("Corpus BLEU:", corpus_bleu)


In [ ]:
# ============================================================
# CELL 8 — BERTSCORE
# model: bert-base-multilingual-cased
# ============================================================

print("Calculating multilingual BERTScore...")

bert_device = "cuda" if torch.cuda.is_available() else "cpu"
bert_batch_size = 8 if torch.cuda.is_available() else 4

P, R, F1 = bert_score(
    eval_df["prediction"].astype(str).tolist(),
    eval_df["gold"].astype(str).tolist(),
    model_type="bert-base-multilingual-cased",
    batch_size=bert_batch_size,
    device=bert_device,
    idf=False,
    rescale_with_baseline=False,
    verbose=True
)

eval_df["BERTScore Precision"] = P.cpu().numpy()
eval_df["BERTScore Recall"] = R.cpu().numpy()
eval_df["BERTScore F1"] = F1.cpu().numpy()

print("BERTScore complete.")


In [ ]:
# ============================================================
# CELL 9 — FINAL RESULT + SAVE prediction.csv AND result.csv
# ============================================================

truncated_count = int(
    eval_df["truncated"]
    .astype(str)
    .str.lower()
    .eq("true")
    .sum()
)

empty_output_count = int(
    eval_df["prediction"].astype(str).str.strip().eq("").sum()
)

avg_generated_tokens = (
    pd.to_numeric(eval_df["generated_tokens"], errors="coerce").mean()
    if "generated_tokens" in eval_df.columns else np.nan
)

max_generated_tokens = (
    pd.to_numeric(eval_df["generated_tokens"], errors="coerce").max()
    if "generated_tokens" in eval_df.columns else np.nan
)

result = pd.DataFrame({
    "metric": [
        "Normalized Exact Match",
        "Token F1",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "METEOR",
        "BERTScore Precision",
        "BERTScore Recall",
        "BERTScore F1",
        "Truncated Outputs",
        "Empty Outputs",
        "Average Generated Tokens",
        "Maximum Generated Tokens",
    ],
    "score": [
        eval_df["Normalized Exact Match"].mean(),
        eval_df["Token F1"].mean(),
        eval_df["Fuzzy Match"].mean(),
        corpus_bleu,
        eval_df["ROUGE-1"].mean(),
        eval_df["ROUGE-2"].mean(),
        eval_df["ROUGE-L"].mean(),
        eval_df["METEOR"].mean(),
        eval_df["BERTScore Precision"].mean(),
        eval_df["BERTScore Recall"].mean(),
        eval_df["BERTScore F1"].mean(),
        truncated_count,
        empty_output_count,
        avg_generated_tokens,
        max_generated_tokens,
    ]
})

# Save row-level predictions + all row-level metrics.
eval_df.to_csv(
    PRED_PATH,
    index=False,
    encoding="utf-8-sig"
)

# Save aggregate metrics.
result.to_csv(
    RESULT_PATH,
    index=False,
    encoding="utf-8-sig"
)

display(result)

print("\nSaved:")
print(PRED_PATH)
print(RESULT_PATH)

print("\nValidation:")
print("Rows evaluated:", len(eval_df))
print("Truncated outputs:", truncated_count)
print("Empty outputs:", empty_output_count)

print("\nNOTE:")
print("/kaggle/input is read-only. Kaggle outputs must be written under /kaggle/working.")
